In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data_raw/Frailty_Study.csv')

In [3]:
df.head()

,Height,Weight,Age,Grip strength,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y


# **a. Unit standardization**

In [4]:
df['Height'] = df['Height'] * 0.0254
df['Weight'] = df['Weight'] * 0.45359237

df = df.rename(columns={'Height': 'Height_m', 'Weight':'Weight_kg'})

In [5]:
df.head()

,Height_m,Weight_kg,Age,Grip strength,Frailty
0,1.67132,50.802345,30,30,N
1,1.81610,61.688562,19,31,N
2,1.76276,69.399633,45,29,N
3,1.73228,64.410117,22,28,Y
4,1.72212,65.317301,29,24,Y


# **b. Feature engineering**

In [6]:
df['BMI'] = np.round(df['Weight_kg'] / (df['Height_m'] ** 2),2)

df.head()

,Height_m,Weight_kg,Age,Grip strength,Frailty,BMI
0,1.67132,50.802345,30,30,N,18.19
1,1.81610,61.688562,19,31,N,18.70
2,1.76276,69.399633,45,29,N,22.33
3,1.73228,64.410117,22,28,Y,21.46
4,1.72212,65.317301,29,24,Y,22.02


In [7]:
bins = [0, 29, 45, 60, np.inf]
labels = ['<30', '30-45', '46-60', '>60']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

df.head()

,Height_m,Weight_kg,Age,Grip strength,Frailty,BMI,AgeGroup
0,1.67132,50.802345,30,30,N,18.19,30-45
1,1.81610,61.688562,19,31,N,18.70,<30
2,1.76276,69.399633,45,29,N,22.33,30-45
3,1.73228,64.410117,22,28,Y,21.46,<30
4,1.72212,65.317301,29,24,Y,22.02,<30


# **c. Categorical → numeric encoding**

In [8]:
df['Frailty'] = df['Frailty'].str.strip().map({'Y': 1, 'N': 0}).astype('int8')

df['Frailty'].dtype

dtype('int8')

In [9]:
df.head()

,Height_m,Weight_kg,Age,Grip strength,Frailty,BMI,AgeGroup
0,1.67132,50.802345,30,30,0,18.19,30-45
1,1.81610,61.688562,19,31,0,18.70,<30
2,1.76276,69.399633,45,29,0,22.33,30-45
3,1.73228,64.410117,22,28,1,21.46,<30
4,1.72212,65.317301,29,24,1,22.02,<30


In [10]:
dummies = pd.get_dummies(df['AgeGroup'], prefix='AgeGroup').astype('int8')
df = pd.concat([df, dummies], axis=1)

df.head()

,Height_m,Weight_kg,Age,Grip strength,Frailty,BMI,AgeGroup,AgeGroup_<30,AgeGroup_30-45,AgeGroup_46-60,AgeGroup_>60
0,1.67132,50.802345,30,30,0,18.19,30-45,0,1,0,0
1,1.81610,61.688562,19,31,0,18.70,<30,1,0,0,0
2,1.76276,69.399633,45,29,0,22.33,30-45,0,1,0,0
3,1.73228,64.410117,22,28,1,21.46,<30,1,0,0,0
4,1.72212,65.317301,29,24,1,22.02,<30,1,0,0,0


# **d. EDA & Reporting**

In [11]:
summary_stats = df.select_dtypes(include=[np.number]).agg(['mean', 'median', 'std']).transpose()
summary_stats

,mean,median,std
Height_m,1.742440,1.738630,0.042435
Weight_kg,59.828834,61.688562,6.455441
Age,32.500000,29.500000,12.860361
Grip strength,26.000000,27.000000,4.521553
Frailty,0.400000,0.000000,0.516398
BMI,19.682000,19.185000,1.780972
AgeGroup_<30,0.500000,0.500000,0.527046
AgeGroup_30-45,0.300000,0.000000,0.483046
AgeGroup_46-60,0.200000,0.000000,0.421637
AgeGroup_>60,0.000000,0.000000,0.000000


In [12]:
findings = summary_stats.to_markdown()

with open('findings.md', 'w') as f:
    f.write(findings)


In [13]:
# Calculate correlation between Grip strength and Frailty
correlation = df['Grip strength'].corr(df['Frailty'])
print(f"Correlation between Grip strength and Frailty: {correlation:.4f}")

Correlation between Grip strength and Frailty: -0.4759


The correlation between Grip strength and Frailty is -0.4759. This show that a moderate negative correlation, suggesting that as grip strength decreases, the likelihood of frailty tends to increase.